<a href="https://colab.research.google.com/github/vivek-chandan/AGentic/blob/main/Agent_currency_exchange.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "sk-Paste_API_KEY_HERE"

In [32]:
from langchain.tools import tool
from langchain_core.tools import Tool
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_core.tools import InjectedToolArg
from typing import Annotated
import requests

In [21]:
!pip install langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 5.1 MB/s eta 0:00:00


In [16]:
#tool
@tool
def get_conversion_factor(base_currency: str, target_currency: str) ->float:
  """
  this function fetches the currency conversion factor from the API between a given base_currency and target_currency
  """
  url = f'https://v6.exchangerate-api.com/v6/da0d1dd4f11cc83c80b7db3c/pair/{base_currency}/{target_currency}'
  respone = requests.get(url)

  return respone.json()

@tool
def convert(base_currency_value:int, conversion_rate: Annotated[float, InjectedToolArg] ) ->float:
  """
  given a currency value and conversion rate this function returns the converted value
  """

  return base_currency_value * conversion_rate

In [6]:
get_conversion_factor.invoke({'base_currency':'USD', 'target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1772928001,
 'time_last_update_utc': 'Sun, 08 Mar 2026 00:00:01 +0000',
 'time_next_update_unix': 1773014401,
 'time_next_update_utc': 'Mon, 09 Mar 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 91.8737}

In [18]:
convert.invoke({'base_currency_value': 3, 'conversion_rate': 91.8737})

275.6211

In [26]:
llm = ChatOpenAI()

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [25]:
#tool binding
llm_with_tools = llm.bind_tools([get_conversion_factor,convert ])

NameError: name 'llm' is not defined

In [29]:

messages = [HumanMessage('what is the  conversion factor between USD and INR and based on the conversion factor can you converse the 20 USD to INR')]

In [34]:
messages

[HumanMessage(content='what is the  conversion factor between USD and INR and based on the conversion factor can you converse the 20 USD to INR', additional_kwargs={}, response_metadata={})]

In [35]:
response = llm_with_tools.invoke(messages)

NameError: name 'llm_with_tools' is not defined

In [33]:
import json

for tool_call in messages.tool_calls:
  # execute the first tool  for conversion rate
  if tool.call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
 #  print(tool_message1)
    # fetch the conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool_message1 to messages
    messages.append(tool_message1)
  # execute the second tool  for convert after the fetch of conversion rate
  if tool.call['name'] == 'convert':
   tool_call['args']['conversion_rate'] = conversion_rate
   tool_message2 = convert.invoke(tool_call)
   messages.append(tool_message2)

AttributeError: 'list' object has no attribute 'tool_calls'

In [ ]:
llm_with_tools.invoke(messages).content